# Task 8 — Database Management (Updated)
**Project:** MSFT Direction Predictor  
 
**Description:** This notebook sets up the SQLite database with 6 tables. It combines the original 10-year datasets with new 2025-2026 data, then filters everything to January 2019 onwards for more relevant and recent market behaviour.

---

## Database Tables
| Table | Filled By | Contents |
|-------|-----------|----------|
| `msft_daily` | This notebook | MSFT OHLCV data (2019-2026) |
| `gold_prices` | This notebook | Daily gold closing price (2019-2026) |
| `oil_prices` | This notebook | Daily crude oil closing price (2019-2026) |
| `vix_data` | This notebook | Daily VIX values (2019-2026) |
| `processed_features` | Pipeline notebook (Task 9) | Engineered features ready for model |
| `predictions` | Model notebook (Task 10) | Model predictions with confidence |

## 1. Import Libraries

In [1]:
import sqlite3
import pandas as pd
import os
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('Libraries imported successfully')

Libraries imported successfully


## 2. Set File Paths

In [2]:
# ── ORIGINAL FILES (2015-2025) ──────────────────────────────────
MSFT_CSV_PATH  = 'data/raw/MSFT.csv'
GOLD_CSV_PATH  = 'data/raw/Gold.csv'
OIL_CSV_PATH   = 'data/raw/CrudeOil.csv'
MACRO_CSV_PATH = 'data/raw/MacroEconomicIndicators.csv'

# ── NEW FILES (2025-2026) ────────────────────────────────────────
MSFT_NEW_PATH  = 'data/raw/Microsoft_2025-2026.csv'
GOLD_NEW_PATH  = 'data/raw/Gold_2025-2026.csv'
OIL_NEW_PATH   = 'data/raw/Oil_2025-2026.csv'
VIX_NEW_PATH   = 'data/raw/VIX_2025-2026.csv'

# ── DATABASE ─────────────────────────────────────────────────────
DB_PATH        = 'data/stocks.db'

# ── DATE FILTER ──────────────────────────────────────────────────
START_DATE     = '2019-01-01'
# ─────────────────────────────────────────────────────────────────

os.makedirs('data/raw', exist_ok=True)

# verify all files exist
all_paths = [
    MSFT_CSV_PATH, GOLD_CSV_PATH, OIL_CSV_PATH, MACRO_CSV_PATH,
    MSFT_NEW_PATH, GOLD_NEW_PATH, OIL_NEW_PATH, VIX_NEW_PATH
]
for path in all_paths:
    assert os.path.exists(path), f'File not found: {path}'
    print(f'  Found: {path}')

print(f'\nAll files verified successfully')
print(f'Date filter: {START_DATE} onwards')

  Found: data/raw/MSFT.csv
  Found: data/raw/Gold.csv
  Found: data/raw/CrudeOil.csv
  Found: data/raw/MacroEconomicIndicators.csv
  Found: data/raw/Microsoft_2025-2026.csv
  Found: data/raw/Gold_2025-2026.csv
  Found: data/raw/Oil_2025-2026.csv
  Found: data/raw/VIX_2025-2026.csv

All files verified successfully
Date filter: 2019-01-01 onwards


## 3. Connect to Database

In [3]:
def create_connection(db_path: str) -> sqlite3.Connection:
    """
    Create a connection to the SQLite database.
    Creates the database file if it does not exist.

    Args:
        db_path (str): Path to the SQLite database file.

    Returns:
        sqlite3.Connection: Active database connection.
    """
    conn = sqlite3.connect(db_path)
    logger.info(f'Connected to database: {db_path}')
    return conn


conn = create_connection(DB_PATH)
print(f'Database connected at: {DB_PATH}')

2026-05-14 11:34:15,436 - INFO - Connected to database: data/stocks.db


Database connected at: data/stocks.db


## 4. Helper — Load New Format CSV
The new 2025-2026 files have a 3-row header that needs to be skipped before reading the actual data.

In [4]:
def load_new_format_csv(csv_path: str, close_col_name: str) -> pd.DataFrame:
    """
    Load a new format CSV file with 3-row header.
    Skips the first 3 rows and reads date + close only.

    Args:
        csv_path (str): Path to the new format CSV file.
        close_col_name (str): Name to give the close column.

    Returns:
        pd.DataFrame: DataFrame with date and close columns.
    """
    # skip first 3 rows and assign column names manually
    df = pd.read_csv(
        csv_path,
        skiprows=3,
        header=None,
        names=['date', 'close', 'high', 'low', 'open', 'volume']
    )

    # drop empty rows
    df = df.dropna(subset=['date'])
    df = df[df['date'].astype(str).str.match(r'\d{4}-\d{2}-\d{2}')]

    # clean date
    df['date'] = pd.to_datetime(df['date']).dt.date

    # keep only date and close
    df = df[['date', 'close']]
    df.columns = ['date', close_col_name]

    df = df.drop_duplicates(subset='date')
    df = df.sort_values('date').reset_index(drop=True)

    return df

## 5. Load and Combine MSFT Data

In [5]:
def load_msft_data(
    conn: sqlite3.Connection,
    original_path: str,
    new_path: str,
    start_date: str
) -> pd.DataFrame:
    """
    Load and combine original and new MSFT data.
    Filters to start_date onwards.

    Args:
        conn (sqlite3.Connection): Active database connection.
        original_path (str): Path to original MSFT CSV.
        new_path (str): Path to new 2025-2026 MSFT CSV.
        start_date (str): Date filter — keep rows from this date onwards.

    Returns:
        pd.DataFrame: Combined and filtered MSFT dataframe.
    """
    # load original
    df_old = pd.read_csv(original_path)
    df_old['Date'] = pd.to_datetime(df_old['Date']).dt.date
    df_old = df_old[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
    df_old.columns = ['date', 'open', 'high', 'low', 'close', 'volume']

    # load new
    df_new = pd.read_csv(
        new_path, skiprows=3, header=None,
        names=['date', 'close', 'high', 'low', 'open', 'volume']
    )
    df_new = df_new.dropna(subset=['date'])
    df_new = df_new[df_new['date'].astype(str).str.match(r'\d{4}-\d{2}-\d{2}')]
    df_new['date'] = pd.to_datetime(df_new['date']).dt.date
    df_new = df_new[['date', 'open', 'high', 'low', 'close', 'volume']]

    # combine
    df = pd.concat([df_old, df_new], ignore_index=True)
    df = df.drop_duplicates(subset='date')
    df = df.sort_values('date').reset_index(drop=True)

    # filter to start date
    df = df[df['date'] >= pd.to_datetime(start_date).date()]
    df = df.reset_index(drop=True)

    df.to_sql('msft_daily', conn, if_exists='replace', index=False)
    logger.info(f'msft_daily loaded: {len(df)} rows')
    return df


msft_df = load_msft_data(conn, MSFT_CSV_PATH, MSFT_NEW_PATH, START_DATE)
print(f'MSFT loaded: {len(msft_df)} rows | {msft_df["date"].min()} to {msft_df["date"].max()}')
msft_df.head(3)

2026-05-14 11:34:15,513 - INFO - msft_daily loaded: 1842 rows


MSFT loaded: 1842 rows | 2019-01-02 to 2026-04-30


,date,open,high,low,close,volume
0,2019-01-02,99.55,101.750,98.94,101.12,35329345.0
1,2019-01-03,100.10,100.185,97.20,97.40,42578410.0
2,2019-01-04,99.72,102.510,98.93,101.93,44060590.0


## 6. Load and Combine Gold Data

In [6]:
def load_gold_data(
    conn: sqlite3.Connection,
    original_path: str,
    new_path: str,
    start_date: str
) -> pd.DataFrame:
    """
    Load and combine original and new Gold price data.
    Filters to start_date onwards.

    Args:
        conn (sqlite3.Connection): Active database connection.
        original_path (str): Path to original Gold CSV.
        new_path (str): Path to new 2025-2026 Gold CSV.
        start_date (str): Date filter.

    Returns:
        pd.DataFrame: Combined and filtered gold dataframe.
    """
    # load original
    df_old = pd.read_csv(original_path)
    df_old['Date'] = pd.to_datetime(df_old['Date'], utc=True).dt.date
    df_old = df_old[['Date', 'Close']]
    df_old.columns = ['date', 'gold_close']

    # load new
    df_new = load_new_format_csv(new_path, 'gold_close')

    # combine
    df = pd.concat([df_old, df_new], ignore_index=True)
    df = df.drop_duplicates(subset='date')
    df = df.sort_values('date').reset_index(drop=True)

    # filter
    df = df[df['date'] >= pd.to_datetime(start_date).date()]
    df = df.reset_index(drop=True)

    df.to_sql('gold_prices', conn, if_exists='replace', index=False)
    logger.info(f'gold_prices loaded: {len(df)} rows')
    return df


gold_df = load_gold_data(conn, GOLD_CSV_PATH, GOLD_NEW_PATH, START_DATE)
print(f'Gold loaded: {len(gold_df)} rows | {gold_df["date"].min()} to {gold_df["date"].max()}')
gold_df.head(3)

2026-05-14 11:34:15,549 - INFO - gold_prices loaded: 1843 rows


Gold loaded: 1843 rows | 2019-01-02 to 2026-04-30


,date,gold_close
0,2019-01-02,1281.000000
1,2019-01-03,1291.800049
2,2019-01-04,1282.699951


## 7. Load and Combine Crude Oil Data

In [7]:
def load_oil_data(
    conn: sqlite3.Connection,
    original_path: str,
    new_path: str,
    start_date: str
) -> pd.DataFrame:
    """
    Load and combine original and new Crude Oil price data.
    Filters to start_date onwards.

    Args:
        conn (sqlite3.Connection): Active database connection.
        original_path (str): Path to original CrudeOil CSV.
        new_path (str): Path to new 2025-2026 Oil CSV.
        start_date (str): Date filter.

    Returns:
        pd.DataFrame: Combined and filtered oil dataframe.
    """
    # load original
    df_old = pd.read_csv(original_path)
    df_old['Date'] = pd.to_datetime(df_old['Date']).dt.date
    df_old = df_old[['Date', 'Close']]
    df_old.columns = ['date', 'oil_close']

    # load new
    df_new = load_new_format_csv(new_path, 'oil_close')

    # combine
    df = pd.concat([df_old, df_new], ignore_index=True)
    df = df.drop_duplicates(subset='date')
    df = df.sort_values('date').reset_index(drop=True)

    # filter
    df = df[df['date'] >= pd.to_datetime(start_date).date()]
    df = df.reset_index(drop=True)

    df.to_sql('oil_prices', conn, if_exists='replace', index=False)
    logger.info(f'oil_prices loaded: {len(df)} rows')
    return df


oil_df = load_oil_data(conn, OIL_CSV_PATH, OIL_NEW_PATH, START_DATE)
print(f'Oil loaded: {len(oil_df)} rows | {oil_df["date"].min()} to {oil_df["date"].max()}')
oil_df.head(3)

2026-05-14 11:34:15,578 - INFO - oil_prices loaded: 1843 rows


Oil loaded: 1843 rows | 2019-01-02 to 2026-04-30


,date,oil_close
0,2019-01-02,46.540001
1,2019-01-03,47.090000
2,2019-01-04,47.959999


## 8. Load and Combine VIX Data

In [8]:
def load_vix_data(
    conn: sqlite3.Connection,
    macro_path: str,
    new_path: str,
    start_date: str
) -> pd.DataFrame:
    """
    Load and combine original VIX from macro file and new 2025-2026 VIX data.
    Filters to start_date onwards.

    Args:
        conn (sqlite3.Connection): Active database connection.
        macro_path (str): Path to MacroEconomicIndicators CSV.
        new_path (str): Path to new 2025-2026 VIX CSV.
        start_date (str): Date filter.

    Returns:
        pd.DataFrame: Combined and filtered VIX dataframe.
    """
    # load original VIX from macro file
    df_old = pd.read_csv(macro_path, index_col=0)
    df_old = df_old[['Stock Market Volatility (VIX Index)']]
    df_old.columns = ['vix']
    df_old = df_old.reset_index()
    df_old.columns = ['date', 'vix']
    df_old['date'] = pd.to_datetime(df_old['date']).dt.date
    df_old = df_old.dropna(subset=['vix'])

    # load new VIX
    df_new = load_new_format_csv(new_path, 'vix')

    # combine
    df = pd.concat([df_old, df_new], ignore_index=True)
    df = df.drop_duplicates(subset='date')
    df = df.sort_values('date').reset_index(drop=True)

    # filter
    df = df[df['date'] >= pd.to_datetime(start_date).date()]
    df = df.reset_index(drop=True)

    df.to_sql('vix_data', conn, if_exists='replace', index=False)
    logger.info(f'vix_data loaded: {len(df)} rows')
    return df


vix_df = load_vix_data(conn, MACRO_CSV_PATH, VIX_NEW_PATH, START_DATE)
print(f'VIX loaded: {len(vix_df)} rows | {vix_df["date"].min()} to {vix_df["date"].max()}')
vix_df.head(3)

2026-05-14 11:34:15,607 - INFO - vix_data loaded: 1863 rows


VIX loaded: 1863 rows | 2019-01-02 to 2026-04-30


,date,vix
0,2019-01-02,23.22
1,2019-01-03,25.45
2,2019-01-04,21.38


## 9. Remove Old Macro Table

In [9]:
# remove old macro_indicators table if it exists
conn2 = sqlite3.connect(DB_PATH)
conn2.execute('DROP TABLE IF EXISTS macro_indicators')
conn2.commit()
conn2.close()
print('macro_indicators table removed')

macro_indicators table removed


## 10. Create Empty Output Tables

In [10]:
def create_output_tables(conn: sqlite3.Connection) -> None:
    """
    Create empty tables for processed features and model predictions.
    These tables are filled by the pipeline and model notebooks.

    Args:
        conn (sqlite3.Connection): Active database connection.
    """
    conn.execute('''
        CREATE TABLE IF NOT EXISTS processed_features (
            date          TEXT PRIMARY KEY,
            open          REAL,
            high          REAL,
            low           REAL,
            close         REAL,
            volume        REAL,
            gold_close    REAL,
            oil_close     REAL,
            vix           REAL,
            lag_1         REAL,
            lag_2         REAL,
            rolling_5     REAL,
            rolling_10    REAL,
            daily_return  REAL,
            price_range   REAL,
            gold_return   REAL,
            oil_return    REAL,
            target        INTEGER
        )
    ''')

    conn.execute('''
        CREATE TABLE IF NOT EXISTS predictions (
            date                TEXT PRIMARY KEY,
            predicted_direction INTEGER,
            confidence          REAL,
            model_used          TEXT,
            actual_direction    INTEGER
        )
    ''')

    conn.commit()
    logger.info('Output tables created')


create_output_tables(conn)
print('Output tables ready')

2026-05-14 11:34:15,644 - INFO - Output tables created


Output tables ready


## 11. Verify All Tables

In [11]:
def verify_database(conn: sqlite3.Connection) -> None:
    """
    Verify all tables exist in the database with correct row counts.

    Args:
        conn (sqlite3.Connection): Active database connection.
    """
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )
    print('Tables in database:')
    for table in tables['name']:
        count = pd.read_sql(f'SELECT COUNT(*) as count FROM {table}', conn)
        rows = count['count'].values[0]
        status = '(empty - fills later)' if rows == 0 else f'{rows} rows'
        print(f'  {table:<25} {status}')


verify_database(conn)

print('\nMissing values check:')
for table, df in [('msft_daily', msft_df), ('gold_prices', gold_df),
                   ('oil_prices', oil_df), ('vix_data', vix_df)]:
    nulls = df.isnull().sum().sum()
    status = '✅ Clean' if nulls == 0 else f'⚠️  {nulls} missing values'
    print(f'  {table:<25} {status}')

Tables in database:
  predictions               (empty - fills later)
  processed_features        1660 rows
  msft_daily                1842 rows
  gold_prices               1843 rows
  oil_prices                1843 rows
  vix_data                  1863 rows

Missing values check:
  msft_daily                ✅ Clean
  gold_prices               ✅ Clean
  oil_prices                ✅ Clean
  vix_data                  ✅ Clean


## 12. Query Functions

In [12]:
def get_msft_data(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query all MSFT daily data from the database."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM msft_daily ORDER BY date', conn)
    conn.close()
    return df


def get_gold_data(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query all Gold price data from the database."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM gold_prices ORDER BY date', conn)
    conn.close()
    return df


def get_oil_data(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query all Crude Oil price data from the database."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM oil_prices ORDER BY date', conn)
    conn.close()
    return df


def get_vix_data(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query all VIX data from the database."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM vix_data ORDER BY date', conn)
    conn.close()
    return df


def get_processed_features(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query processed features from the database."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM processed_features ORDER BY date', conn)
    conn.close()
    return df


def get_predictions(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query all predictions from the database."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM predictions ORDER BY date', conn)
    conn.close()
    return df


def save_processed_features(df: pd.DataFrame, db_path: str = DB_PATH) -> None:
    """
    Save processed features to the database.
    Called by the pipeline notebook after preprocessing.

    Args:
        df (pd.DataFrame): Processed features dataframe.
        db_path (str): Path to the SQLite database.
    """
    conn = sqlite3.connect(db_path)
    df.to_sql('processed_features', conn, if_exists='replace', index=False)
    conn.close()
    logger.info(f'Saved {len(df)} rows to processed_features')


def save_predictions(df: pd.DataFrame, db_path: str = DB_PATH) -> None:
    """
    Save model predictions to the database.
    Called by the model notebook after predictions are made.

    Args:
        df (pd.DataFrame): Predictions dataframe.
        db_path (str): Path to the SQLite database.
    """
    conn = sqlite3.connect(db_path)
    df.to_sql('predictions', conn, if_exists='replace', index=False)
    conn.close()
    logger.info(f'Saved {len(df)} rows to predictions')


# test all functions
print('Testing query functions:')
print(f'  get_msft_data()           → {len(get_msft_data())} rows')
print(f'  get_gold_data()           → {len(get_gold_data())} rows')
print(f'  get_oil_data()            → {len(get_oil_data())} rows')
print(f'  get_vix_data()            → {len(get_vix_data())} rows')
print(f'  get_processed_features()  → {len(get_processed_features())} rows (empty - fills after Task 9)')
print(f'  get_predictions()         → {len(get_predictions())} rows (empty - fills after Task 10)')

Testing query functions:
  get_msft_data()           → 1842 rows
  get_gold_data()           → 1843 rows
  get_oil_data()            → 1843 rows
  get_vix_data()            → 1863 rows
  get_processed_features()  → 1660 rows (empty - fills after Task 9)
  get_predictions()         → 0 rows (empty - fills after Task 10)


## 13. Close Connection

In [13]:
conn.close()
print('Database connection closed.')
print(f'\nDatabase saved at: {DB_PATH}')
print(f'Date range: {START_DATE} to present')
print('\nTask 8 complete — 6 tables ready:')
print('  Raw data  : msft_daily, gold_prices, oil_prices, vix_data')
print('  Output    : processed_features (empty), predictions (empty)')
print('\nNext: Run Task 9 pipeline notebook to fill processed_features')

Database connection closed.

Database saved at: data/stocks.db
Date range: 2019-01-01 to present

Task 8 complete — 6 tables ready:
  Raw data  : msft_daily, gold_prices, oil_prices, vix_data
  Output    : processed_features (empty), predictions (empty)

Next: Run Task 9 pipeline notebook to fill processed_features
